# Digital Echoes vs. Bed Nights
## Predicting Viral Tourism Demand Shocks from Digital Signals

**DTU 42578 Advanced Business Analytics — Spring 2026**  
*Group: Alex (Spain · ES), Beatriz (Croatia · HR), Cesia (Italy · IT & Albania · AL), Frede (Malta · MT & Data Engineering)*

---

## 1. Introduction

Viral social-media content can trigger rapid, hard-to-predict surges in tourism demand — so-called *viral tourism shocks*. This report investigates whether **digital attention signals** (Google Trends, Wikipedia pageviews) measured *before* a surge carry sufficient predictive power to forecast Eurostat bed-night arrivals at the regional level across six EU destinations: Spain, Croatia, Italy, Albania, Malta, and Greece.

**Research question:** Do digital signals predict tourist arrivals with a measurable lead time, and can we build a resilience-aware system to help destinations prepare?

**Methods (≥ 3 course topics):**
| Method | Course topic |
|--------|-------------|
| Random Forest + SHAP | Forecasting · Explainable AI |
| Graph Neural Network (GNN) | GNNs — spatial spillover |
| Recommender System | Recommender systems |
| Granger causality + COVID policy controls | Causal methods · Endogeneity |
| Seasonality decomposition (month + YoY) | Time-series methods |

---
## 2. Data Collection


### 2.0 Setup & Imports


In [ ]:
import os, sys, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})

# ── Install dependencies ──────────────────────────────────────────────────────
os.system('pip install pytrends eurostat openpyxl -q')

# ── Clone repo — contains pre-collected raw files & source data ───────────────
REPO = 'viral-tourism-resilience'
if not os.path.exists(REPO):
    os.system(f'git clone https://github.com/cniq182/viral-tourism-resilience.git')

RAW_DIR  = f'{REPO}/data/raw'
PROC_DIR = f'{REPO}/data/processed'

# ── Pipeline flag ─────────────────────────────────────────────────────────────
# FORCE_RECOLLECT = True  → re-run all API calls from scratch (~1-2 hours)
# FORCE_RECOLLECT = False → load pre-collected files from the cloned repo
FORCE_RECOLLECT = False

print('Repo found:', os.path.exists(REPO))
print('FORCE_RECOLLECT:', FORCE_RECOLLECT)


### 2.1 Google Trends — Worldwide Collection

We collect 6 Google Trends signals per city (airbnb, hotel, flights, 3 landmark attractions) using the [pytrends](https://github.com/GeneralMills/pytrends) library with `geo=''` (worldwide, not country-filtered) to capture **international tourist intent** rather than domestic searches.  

**Normalisation:** pytrends caps requests at 5 keywords per call. We use an *anchor term* (attraction_2) that appears in both batches to cross-scale attraction_3 onto the same 0–100 index as the other 5 signals.  

> ⏱ `FORCE_RECOLLECT = True` takes ~1–2 hours due to API rate-limit back-offs. Pre-collected CSVs are included in the repo.


In [ ]:
import pathlib

# ── urllib3 v2 compatibility patch ────────────────────────────────────────────
import urllib3.util.retry as _retry_module
if not hasattr(_retry_module.Retry.__init__, '_patched'):
    _orig = _retry_module.Retry.__init__
    def _p(self, *a, **kw): kw.pop('method_whitelist', None); _orig(self, *a, **kw)
    _p._patched = True
    _retry_module.Retry.__init__ = _p
from pytrends.request import TrendReq

GT_COLS   = ['gt_airbnb','gt_hotel','gt_flights','gt_attraction_1','gt_attraction_2','gt_attraction_3']
TIMEFRAME = '2020-01-01 2026-03-31'
GEO       = ''  # worldwide

ALL_CITIES = {
    'barcelona': {'country':'Spain','region':'Cataluña',
        'gt_airbnb':'Barcelona airbnb','gt_hotel':'Barcelona hotel','gt_flights':'Barcelona flights',
        'gt_attraction_1':'Sagrada Familia','gt_attraction_2':'Park Guell','gt_attraction_3':'Barcelona festival'},
    'madrid': {'country':'Spain','region':'Comunidad de Madrid',
        'gt_airbnb':'Madrid airbnb','gt_hotel':'Madrid hotel','gt_flights':'Madrid flights',
        'gt_attraction_1':'Prado Museum','gt_attraction_2':'Retiro Park','gt_attraction_3':'Madrid festival'},
    'valencia': {'country':'Spain','region':'Comunitat Valenciana',
        'gt_airbnb':'Valencia airbnb','gt_hotel':'Valencia hotel','gt_flights':'Valencia flights',
        'gt_attraction_1':'City of Arts and Sciences','gt_attraction_2':'Valencia Cathedral','gt_attraction_3':'Las Fallas'},
    'balearics': {'country':'Spain','region':'Illes Balears',
        'gt_airbnb':'Mallorca airbnb','gt_hotel':'Mallorca hotel','gt_flights':'Mallorca flights',
        'gt_attraction_1':'Palma de Mallorca','gt_attraction_2':'Formentera','gt_attraction_3':'Ibiza festival'},
    'andalusia': {'country':'Spain','region':'Andalucía',
        'gt_airbnb':'Seville airbnb','gt_hotel':'Seville hotel','gt_flights':'Malaga flights',
        'gt_attraction_1':'Alhambra','gt_attraction_2':'Seville Cathedral','gt_attraction_3':'Feria de Abril'},
    'canary': {'country':'Spain','region':'Canarias',
        'gt_airbnb':'Tenerife airbnb','gt_hotel':'Tenerife hotel','gt_flights':'Tenerife flights',
        'gt_attraction_1':'Teide volcano','gt_attraction_2':'Gran Canaria beaches','gt_attraction_3':'Canary Islands festival'},
    'split': {'country':'Croatia','region':'Jadranska Hrvatska',
        'gt_airbnb':'Split Croatia airbnb','gt_hotel':'Split Croatia hotel','gt_flights':'Split Croatia flights',
        'gt_attraction_1':'Diocletian Palace','gt_attraction_2':'Marjan Hill','gt_attraction_3':'Split Riva'},
    'dubrovnik': {'country':'Croatia','region':'Jadranska Hrvatska',
        'gt_airbnb':'Dubrovnik airbnb','gt_hotel':'Dubrovnik hotel','gt_flights':'Dubrovnik flights',
        'gt_attraction_1':'Dubrovnik Old Town','gt_attraction_2':'Walls of Dubrovnik','gt_attraction_3':'Lokrum Island'},
    'osijek': {'country':'Croatia','region':'Panonska Hrvatska',
        'gt_airbnb':'Osijek Croatia airbnb','gt_hotel':'Osijek Croatia hotel','gt_flights':'Osijek Croatia flights',
        'gt_attraction_1':'Tvrdja Osijek','gt_attraction_2':'Kopacki Rit','gt_attraction_3':'Drava River Osijek'},
    'varazdin': {'country':'Croatia','region':'Sjeverna Hrvatska',
        'gt_airbnb':'Varazdin Croatia airbnb','gt_hotel':'Varazdin Croatia hotel','gt_flights':'Varazdin Croatia flights',
        'gt_attraction_1':'Varazdin Castle','gt_attraction_2':'Spancirfest','gt_attraction_3':'Varazdin Cemetery'},
    'zagreb': {'country':'Croatia','region':'Grad Zagreb',
        'gt_airbnb':'Zagreb airbnb','gt_hotel':'Zagreb hotel','gt_flights':'Zagreb flights',
        'gt_attraction_1':'Zagreb Cathedral','gt_attraction_2':'Ban Jelacic Square','gt_attraction_3':'Maksimir Park'},
    'milan': {'country':'Italy','region':'Lombardia',
        'gt_airbnb':'Milan airbnb','gt_hotel':'Milan hotel','gt_flights':'Milan flights',
        'gt_attraction_1':'Duomo di Milano','gt_attraction_2':'Galleria Vittorio Emanuele II','gt_attraction_3':'Teatro alla Scala'},
    'venice': {'country':'Italy','region':'Veneto',
        'gt_airbnb':'Venice airbnb','gt_hotel':'Venice hotel','gt_flights':'Venice flights',
        'gt_attraction_1':"St Mark's Basilica",'gt_attraction_2':'Grand Canal Venice','gt_attraction_3':"Doge's Palace"},
    'rome': {'country':'Italy','region':'Lazio',
        'gt_airbnb':'Rome airbnb','gt_hotel':'Rome hotel','gt_flights':'Rome flights',
        'gt_attraction_1':'Colosseum','gt_attraction_2':'Trevi Fountain','gt_attraction_3':'Pantheon Rome'},
    'naples': {'country':'Italy','region':'Campania',
        'gt_airbnb':'Naples Italy airbnb','gt_hotel':'Naples Italy hotel','gt_flights':'Naples Italy flights',
        'gt_attraction_1':'Mount Vesuvius','gt_attraction_2':'Pompeii','gt_attraction_3':'Amalfi Coast'},
    'palermo': {'country':'Italy','region':'Sicilia',
        'gt_airbnb':'Palermo airbnb','gt_hotel':'Palermo hotel','gt_flights':'Palermo flights',
        'gt_attraction_1':'Valley of the Temples','gt_attraction_2':'Mount Etna','gt_attraction_3':'Palermo Cathedral'},
    'tirana': {'country':'Albania','region':'Qender',
        'gt_airbnb':'Tirana airbnb','gt_hotel':'Tirana hotel','gt_flights':'Tirana flights',
        'gt_attraction_1':'Skanderbeg Square','gt_attraction_2':'Bunk Art Museum','gt_attraction_3':'Et hem Bey Mosque'},
    'sarande': {'country':'Albania','region':'Jug',
        'gt_airbnb':'Sarande airbnb','gt_hotel':'Sarande hotel','gt_flights':'Sarande Albania flights',
        'gt_attraction_1':'Ksamil Beach','gt_attraction_2':'Butrint','gt_attraction_3':'Blue Eye Spring Albania'},
    'berat': {'country':'Albania','region':'Qender',
        'gt_airbnb':'Berat airbnb','gt_hotel':'Berat hotel','gt_flights':'Berat Albania flights',
        'gt_attraction_1':'Berat Castle','gt_attraction_2':'Onufri Museum','gt_attraction_3':'Gorica Bridge Berat'},
    'shkoder': {'country':'Albania','region':'Veri',
        'gt_airbnb':'Shkoder airbnb','gt_hotel':'Shkoder hotel','gt_flights':'Shkoder Albania flights',
        'gt_attraction_1':'Shkoder Castle','gt_attraction_2':'Lake Shkoder','gt_attraction_3':'Marubi Museum'},
    'gjirokastra': {'country':'Albania','region':'Jug',
        'gt_airbnb':'Gjirokastra airbnb','gt_hotel':'Gjirokastra hotel','gt_flights':'Gjirokastra Albania',
        'gt_attraction_1':'Gjirokaster Castle','gt_attraction_2':'Old Bazaar Gjirokastra','gt_attraction_3':'Ali Pasha Tower'},
    'valletta': {'country':'Malta','region':'Malta',
        'gt_airbnb':'Valletta airbnb','gt_hotel':'Valletta hotel','gt_flights':'Malta flights',
        'gt_attraction_1':'Three Cities Malta','gt_attraction_2':'Grand Harbour Malta','gt_attraction_3':'Fort Saint Elmo Malta'},
    'st_julians': {'country':'Malta','region':'Malta',
        'gt_airbnb':'St Julians Malta airbnb','gt_hotel':'St Julians Malta hotel','gt_flights':'Malta flights',
        'gt_attraction_1':'St John Co-Cathedral Malta','gt_attraction_2':'Spinola Bay Malta','gt_attraction_3':'Rotunda of Mosta'},
    'st_pauls_bay': {'country':'Malta','region':'Malta',
        'gt_airbnb':'St Pauls Bay Malta airbnb','gt_hotel':'St Pauls Bay Malta hotel','gt_flights':'Malta flights',
        'gt_attraction_1':'Ghadira Bay Malta','gt_attraction_2':'Popeye Village Malta','gt_attraction_3':'Golden Bay Malta'},
    'victoria_gozo': {'country':'Malta','region':'Malta',
        'gt_airbnb':'Gozo airbnb','gt_hotel':'Gozo hotel','gt_flights':'Gozo Malta flights',
        'gt_attraction_1':'Citadella Gozo','gt_attraction_2':'Dwejra Gozo','gt_attraction_3':'Blue Lagoon Malta'},
    'athens': {'country':'Greece','region':'Attiki',
        'gt_airbnb':'Athens airbnb','gt_hotel':'Athens hotel','gt_flights':'Athens flights',
        'gt_attraction_1':'Acropolis Athens','gt_attraction_2':'Parthenon','gt_attraction_3':'Cape Sounion'},
    'rhodes': {'country':'Greece','region':'Notio Aigaio',
        'gt_airbnb':'Rhodes airbnb','gt_hotel':'Rhodes hotel','gt_flights':'Rhodes flights',
        'gt_attraction_1':'Medieval City of Rhodes','gt_attraction_2':'Santorini','gt_attraction_3':'Delos archaeological site'},
    'heraklion': {'country':'Greece','region':'Kriti',
        'gt_airbnb':'Heraklion airbnb','gt_hotel':'Heraklion hotel','gt_flights':'Heraklion flights',
        'gt_attraction_1':'Palace of Knossos','gt_attraction_2':'Samaria Gorge','gt_attraction_3':'Elafonissi Beach'},
    'thessaloniki': {'country':'Greece','region':'Kentriki Makedonia',
        'gt_airbnb':'Thessaloniki airbnb','gt_hotel':'Thessaloniki hotel','gt_flights':'Thessaloniki flights',
        'gt_attraction_1':'White Tower Thessaloniki','gt_attraction_2':'Mount Olympus','gt_attraction_3':'Vergina archaeological site'},
    'corfu': {'country':'Greece','region':'Ionia Nisia',
        'gt_airbnb':'Corfu airbnb','gt_hotel':'Corfu hotel','gt_flights':'Corfu flights',
        'gt_attraction_1':'Old Town of Corfu','gt_attraction_2':'Navagio Beach Zakynthos','gt_attraction_3':'Melissani Cave Kefalonia'},
}

if FORCE_RECOLLECT:
    pytrends = TrendReq(hl='en-US', tz=0, timeout=(10,25), retries=2, backoff_factor=0.3)
    os.makedirs(RAW_DIR, exist_ok=True)

    def fetch_batch(city_key, label, batch, max_empty=3, max_ratelim=5):
        for attempt in range(max_ratelim):
            try:
                pytrends.build_payload(batch, timeframe=TIMEFRAME, geo=GEO)
                df = pytrends.interest_over_time()
                if df is None or df.empty:
                    if attempt >= max_empty - 1:
                        print(f'  [{city_key}] {label} attempt {attempt+1} empty — giving up'); return None
                    wait = (2**attempt)*random.uniform(12,25)
                    print(f'  [{city_key}] {label} attempt {attempt+1} empty — waiting {wait:.0f}s'); time.sleep(wait); continue
                time.sleep(random.uniform(8,15)); print(f'  [{city_key}] {label} ok'); return df
            except Exception as e:
                if attempt >= max_ratelim-1:
                    print(f'  [{city_key}] {label} failed: {str(e)[:50]} — giving up'); return None
                wait = (2**attempt)*random.uniform(15,30)
                print(f'  [{city_key}] {label} failed: {str(e)[:50]} — waiting {wait:.0f}s'); time.sleep(wait)
        return None

    def fetch_city(city_key, cfg):
        out_path = pathlib.Path(RAW_DIR) / f'gt_worldwide_{city_key}.csv'
        if out_path.exists(): print(f'  [{city_key}] cached — skipping'); return
        queries = [cfg[c] for c in GT_COLS]
        q_airbnb,q_hotel,q_flights,q_att1,q_att2,q_att3 = queries
        df1 = fetch_batch(city_key, 'batch1', [q_airbnb,q_hotel,q_flights,q_att1,q_att2])
        df2 = fetch_batch(city_key, 'batch2', [q_att2, q_att3])
        if df1 is None: return
        if df1 is not None and df2 is not None:
            scale = df1[q_att2].replace(0,np.nan) / df2[q_att2].replace(0,np.nan)
            att3  = (df2[q_att3]*scale).round().clip(0,100)
        else:
            att3 = pd.Series(0, index=df1.index)
        result = pd.DataFrame({'gt_airbnb':df1[q_airbnb],'gt_hotel':df1[q_hotel],
            'gt_flights':df1[q_flights],'gt_attraction_1':df1[q_att1],
            'gt_attraction_2':df1[q_att2],'gt_attraction_3':att3}, index=df1.index)
        result = result.reset_index()
        result.insert(0,'city',city_key); result.insert(0,'region',cfg['region']); result.insert(0,'country',cfg['country'])
        result['date'] = pd.to_datetime(result['date']).dt.to_period('M').dt.to_timestamp()
        result = result[result['date']>='2020-01-01'].reset_index(drop=True)
        result[GT_COLS] = result[GT_COLS].fillna(0).astype(int)
        result.to_csv(out_path, index=False)
        print(f'  [{city_key}] saved {len(result)} rows')

    print(f'Collecting GT for {len(ALL_CITIES)} cities worldwide...')
    for i,(city_key,cfg) in enumerate(ALL_CITIES.items()):
        print(f'[{i+1}/{len(ALL_CITIES)}] {city_key}')
        fetch_city(city_key, cfg)
        if i < len(ALL_CITIES)-1: time.sleep(random.uniform(20,35))
    print('GT collection done.')

# ── Load GT from pre-collected CSVs ───────────────────────────────────────────
gt_frames = []
for city_key, cfg in ALL_CITIES.items():
    p = pathlib.Path(RAW_DIR) / f'gt_worldwide_{city_key}.csv'
    if p.exists():
        gt_frames.append(pd.read_csv(p, parse_dates=['date']))
    else:
        print(f'WARNING: {p.name} missing')
gt_df = pd.concat(gt_frames, ignore_index=True)
gt_df['date'] = pd.to_datetime(gt_df['date']).dt.to_period('M').dt.to_timestamp()
print(f'GT loaded: {gt_df.shape}  — cities: {gt_df["city"].nunique()}')
gt_df.head(3)


### 2.2 Eurostat — NUTS2 Monthly Nights Spent (`tour_occ_nin2m`)

The dataset `tour_occ_nin2m` provides **monthly nights spent by tourists at NUTS2 regional level** (2020–2024). It is distributed as an Excel workbook with one sheet per year; month values are at every odd column index (1, 3, 5, …, 23) with flag columns in between.


In [ ]:
NUTS2_XLSX = f'{REPO}/data/raw/tour_occ_nin2m.xlsx'

# NUTS2 region label → list of city keys in our dataset
REGION_CITY = {
    'Comunidad de Madrid':  ['madrid'],
    'Cataluña':             ['barcelona'],
    'Comunitat Valenciana': ['valencia'],
    'Illes Balears':        ['balearics'],
    'Andalucía':            ['andalusia'],
    'Canarias':             ['canary'],
    'Jadranska Hrvatska':   ['split','dubrovnik'],
    'Panonska Hrvatska':    ['osijek'],
    'Grad Zagreb':          ['zagreb'],
    'Sjeverna Hrvatska':    ['varazdin'],
    'Lombardia':            ['milan'],
    'Veneto':               ['venice'],
    'Lazio':                ['rome'],
    'Campania':             ['naples'],
    'Sicilia':              ['palermo'],
    'Veri':                 ['shkoder'],
    'Qender':               ['tirana','berat'],
    'Jug':                  ['sarande','gjirokastra'],
    'Attiki':               ['athens'],
    'Notio Aigaio':         ['rhodes'],
    'Kriti':                ['heraklion'],
    'Kentriki Makedonia':   ['thessaloniki'],
    'Ionia Nisia':          ['corfu'],
    'Malta':                ['valletta','st_julians','st_pauls_bay','victoria_gozo'],
}
CITY_REGION = {city: region for region, cities in REGION_CITY.items() for city in cities}

# Parse Excel: 5 sheets (years 2020-2024), month values at odd column indices
YEAR_SHEETS = {1:2020, 2:2021, 3:2022, 4:2023, 5:2024}
MONTH_COLS  = list(range(1, 24, 2))  # indices 1,3,5,...,23 → Jan–Dec

nuts2_records = []
xls = pd.ExcelFile(NUTS2_XLSX)
for sheet_num, year in YEAR_SHEETS.items():
    raw = pd.read_excel(xls, sheet_name=sheet_num-1, header=None)
    for _, row in raw.iterrows():
        region_label = str(row.iloc[0]).strip()
        if region_label not in REGION_CITY:
            continue
        for month_idx, col in enumerate(MONTH_COLS, start=1):
            val = row.iloc[col]
            nights = pd.to_numeric(val, errors='coerce') if str(val) not in (':', 'nan', '') else np.nan
            date   = pd.Timestamp(year=year, month=month_idx, day=1)
            for city in REGION_CITY[region_label]:
                nuts2_records.append({'date': date, 'city': city, 'nights_spent_nuts2': nights})

nuts2_df = pd.DataFrame(nuts2_records)
print(f'NUTS2 nights parsed: {nuts2_df.shape}')
print(f'NaN rate: {nuts2_df["nights_spent_nuts2"].isna().mean():.1%}')
nuts2_df.head(3)


### 2.3 Eurostat — Country-Level Arrivals & Bed Places

We pull three additional Eurostat datasets via the `eurostat` Python package:  
- **`tour_occ_arm`** — monthly arrivals at tourist accommodation (country level)  
- **`tour_occ_nim`** — monthly nights spent at country level  
- **`tour_cap_nat`** — annual bed places (capacity), from which we derive `avg_length_of_stay` and `occupancy_rate`


In [ ]:
import eurostat

COUNTRIES  = ['ES','HR','IT','AL','MT','EL']
NAME_MAP   = {'ES':'Spain','HR':'Croatia','IT':'Italy','AL':'Albania','MT':'Malta','EL':'Greece'}
DATE_START = '2020-01'

def fetch_eurostat_long(dataset_code, value_col, filters):
    """Fetch Eurostat dataset and return filtered long-format DataFrame."""
    raw = eurostat.get_data_df(dataset_code)
    # First column contains pipe-separated filter keys
    key_col = raw.columns[0]
    for key, val in filters.items():
        raw = raw[raw[key_col].str.contains(rf'(?:^|[,\\])(?:{"|".join(val) if isinstance(val,list) else val})(?:[,\\]|$)', regex=True)]
    # Melt date columns
    date_cols = [c for c in raw.columns if str(c).startswith('20') or str(c).startswith('19')]
    melted = raw.melt(id_vars=[key_col], value_vars=date_cols, var_name='date', value_name=value_col)
    melted['geo'] = melted[key_col].str.split(r'[,\\]').str[-1].str.strip()
    melted = melted[melted['geo'].isin(COUNTRIES)].copy()
    melted['date'] = pd.to_datetime(melted['date'], errors='coerce')
    melted = melted.dropna(subset=['date'])
    melted[value_col] = pd.to_numeric(melted[value_col], errors='coerce')
    melted['country'] = melted['geo'].map(NAME_MAP)
    return melted[['date','country',value_col]].dropna(subset=['country'])

print('Fetching tour_occ_arm (monthly arrivals)...')
arrivals_raw = fetch_eurostat_long(
    'tour_occ_arm', 'arrivals_country',
    {'c_resid':'TOTAL', 'unit':'NR', 'nace_r2':'I551-I553'}
)
arrivals_df = arrivals_raw[arrivals_raw['date'] >= DATE_START].groupby(['date','country'], as_index=False)['arrivals_country'].sum()

print('Fetching tour_occ_nim (monthly nights, country)...')
nights_raw = fetch_eurostat_long(
    'tour_occ_nim', 'nights_spent_country',
    {'c_resid':'TOTAL', 'unit':'NR', 'nace_r2':'I551-I553'}
)
nights_country_df = nights_raw[nights_raw['date'] >= DATE_START].groupby(['date','country'], as_index=False)['nights_spent_country'].sum()

print('Fetching tour_cap_nat (annual bed places)...')
beds_raw = fetch_eurostat_long(
    'tour_cap_nat', 'bed_places_country',
    {'unit':'NR', 'nace_r2':'I551-I553'}
)
beds_df = beds_raw.copy()
beds_df['year'] = beds_df['date'].dt.year
beds_annual = beds_df.groupby(['year','country'], as_index=False)['bed_places_country'].mean()

print(f'arrivals_df: {arrivals_df.shape}, nights_country_df: {nights_country_df.shape}, beds_annual: {beds_annual.shape}')


### 2.4 COVID-19 Policy Stringency Controls

To address endogeneity between digital signals and actual arrivals during the pandemic, we merge three OxCGRT indicators as control variables — including travel-ban stringency for the **top 3 origin markets** of each destination country.

*Source: [Oxford COVID-19 Government Response Tracker v4](https://github.com/OxCGRT/covid-policy-dataset)*


In [ ]:
OXCGRT_FILE = f'{REPO}/data/processed/oxcgrt_covid_data.csv'

TOP_ORIGINS = {
    'Albania': ['Kosovo','North Macedonia','Italy'],
    'Croatia': ['Germany','Slovenia','Austria'],
    'Greece':  ['United Kingdom','Germany','France'],
    'Italy':   ['Germany','United States','United Kingdom'],
    'Malta':   ['United Kingdom','Italy','France'],
    'Spain':   ['United Kingdom','France','Germany'],
}

ox = pd.read_csv(OXCGRT_FILE, usecols=[
    'CountryName','Date',
    'C8EV_International travel controls',
    'C6M_Stay at home requirements',
    'C4M_Restrictions on gatherings'
], low_memory=False)
ox['Date'] = pd.to_datetime(ox['Date'].astype(str), format='%Y%m%d')
ox['year_month'] = ox['Date'].values.astype('datetime64[M]')
ox_m = ox.groupby(['CountryName','year_month'], as_index=False).mean(numeric_only=True)

# Destination stringency
dest_ox = ox_m[['CountryName','year_month',
    'C6M_Stay at home requirements',
    'C4M_Restrictions on gatherings']].rename(columns={
    'C6M_Stay at home requirements':  'dest_stay_at_home',
    'C4M_Restrictions on gatherings': 'dest_gathering_restrictions',
})

# Origin travel-ban lookup
travel_lkp = ox_m.set_index(['CountryName','year_month'])['C8EV_International travel controls'].to_dict()

print(f'OxCGRT loaded: {ox_m.shape[0]} country-month observations')
print('Stringency controls ready.')


### 2.5 Dataset Assembly


In [ ]:
# 1. Start from GT signals (one row per city × month)
master = gt_df[['date','country','city'] + GT_COLS].copy()

# 2. Add region (NUTS2 label)
master['region'] = master['city'].map(CITY_REGION)

# 3. Merge NUTS2 monthly nights (2020-2024, NaN for 2025-2026)
master = master.merge(nuts2_df, on=['date','city'], how='left')

# 4. Merge country-level nights
master = master.merge(nights_country_df, on=['date','country'], how='left')

# 5. Merge monthly arrivals
master = master.merge(arrivals_df, on=['date','country'], how='left')

# 6. Merge annual bed places (join on year)
master['year'] = master['date'].dt.year
master = master.merge(beds_annual, on=['year','country'], how='left')
master.drop(columns=['year'], inplace=True)

# 7. Derive avg length of stay and occupancy rate
master['avg_length_of_stay_country'] = master['nights_spent_country'] / master['arrivals_country'].replace(0, np.nan)
days_in_month = master['date'].dt.days_in_month
master['occupancy_rate_country'] = (
    master['nights_spent_country'] / (master['bed_places_country'] * days_in_month)
).clip(0, 1)

# 8. Merge COVID destination controls
master = master.merge(dest_ox, how='left', left_on=['country','date'], right_on=['CountryName','year_month'])
master.drop(columns=['CountryName','year_month'], inplace=True)

# 9. Merge origin travel-ban (top 3)
def _origin_bans(row):
    origins = TOP_ORIGINS.get(row['country'], [])
    vals = [travel_lkp.get((o, row['date'])) for o in origins]
    vals = [v for v in vals if v is not None]
    return pd.Series({
        'origin1_stringency': travel_lkp.get((origins[0], row['date']), 0) if len(origins)>0 else 0,
        'origin2_stringency': travel_lkp.get((origins[1], row['date']), 0) if len(origins)>1 else 0,
        'origin3_stringency': travel_lkp.get((origins[2], row['date']), 0) if len(origins)>2 else 0,
        'origin_top3_stringency_avg': sum(vals)/len(vals) if vals else 0,
    })
master[['origin1_stringency','origin2_stringency','origin3_stringency','origin_top3_stringency_avg']] = \
    master.apply(_origin_bans, axis=1)

# Fill COVID NaN (pre/post pandemic) with 0
covid_cols = ['dest_stay_at_home','dest_gathering_restrictions',
              'origin1_stringency','origin2_stringency','origin3_stringency','origin_top3_stringency_avg']
master[covid_cols] = master[covid_cols].fillna(0)

# Rename dest stringency columns to match existing master
master.rename(columns={
    'dest_stay_at_home':            'dest_stringency_index',
    'dest_gathering_restrictions':  'dest_gathering_restrictions',
}, inplace=True)

master.sort_values(['country','region','city','date'], inplace=True)
master.reset_index(drop=True, inplace=True)

print(f'Assembly complete: {master.shape}')
print('Columns:', list(master.columns))


### 2.6 Seasonality Features


In [ ]:
GEO_KEYS = ['country','region','city']

master['month'] = master['date'].dt.month

master['arrivals_last_year'] = (
    master.groupby(GEO_KEYS)['arrivals_country'].shift(12)
)

master['arrivals_yoy_pct_change'] = (
    (master['arrivals_country'] - master['arrivals_last_year'])
    / master['arrivals_last_year'].replace(0, np.nan) * 100
)

print('Seasonality features added.')
print(f'arrivals_last_year non-null: {master["arrivals_last_year"].notna().sum()}/{len(master)}')
print(f'  (first 12 months per city are NaN by design)')


### 2.7 Final Dataset


In [ ]:
# Save
os.makedirs(PROC_DIR, exist_ok=True)
out_path = f'{PROC_DIR}/master_tourism_dataset.csv'
master.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

# Overview
df = master.copy()  # df used in all subsequent sections
print(f'Shape: {df.shape}')
print(f'Countries: {sorted(df["country"].unique())}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Columns: {list(df.columns)}')
print()
print('NaN summary:')
print(df.isna().sum()[df.isna().sum()>0])
df.head()


---
## 3. Dataset Pre-processing & Visualisation

### 3.1 Seasonality Features

Three features capture the seasonal structure of tourism demand:  
- `month` — calendar month (1–12) used as a cyclic control  
- `arrivals_last_year` — 12-month lag of `arrivals_country` within each geographic unit  
- `arrivals_yoy_pct_change` — year-over-year percentage change  

*Script adapted from `FINAL/add_seasonality.py` (Frede's branch).*

*Seasonality features (`month`, `arrivals_last_year`, `arrivals_yoy_pct_change`) are computed in §2.6 above and available in `df`.*


### 3.2 Endogeneity Controls — COVID Policy Stringency

To address endogeneity between digital signals and actual arrivals during the pandemic, we merge three OxCGRT indicators as control variables:

| Feature | OxCGRT indicator | Interpretation |
|---------|-----------------|----------------|
| `dest_stay_at_home` | C6M | Destination stay-at-home stringency |
| `dest_gathering_restrictions` | C4M | Restrictions on gatherings at destination |
| `origin_top3_travel_ban_avg` | C8EV (avg. top-3 origin countries) | International travel ban from main sending markets |

*Script adapted from `FINAL/build_dataset_custom_covid.py` (Frede's branch).*

*COVID stringency controls (`dest_stringency_index`, `origin1/2/3_stringency`, `origin_top3_stringency_avg`) are merged in §2.4–2.5 above and available in `df`.*


### 3.3 Exploratory Data Analysis & Visualisation

> Each team member adds plots for their own region(s) below.

In [ ]:
# ── Spain — EDA (Alex) ────────────────────────────────────────────────────────
# TODO (Alex): time-series plot of GT signals vs. nights_spent for ES regions,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass

In [ ]:
# ── Croatia — EDA (Beatriz) ───────────────────────────────────────────────────
# TODO (Beatriz): time-series, shock markers, correlation heatmap for HR.
pass

In [ ]:
# ── Italy — EDA (Cesia) ───────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for IT.
pass

In [ ]:
# ── Albania — EDA (Cesia) ─────────────────────────────────────────────────────
# TODO (Cesia): time-series, shock markers, correlation heatmap for AL.
pass

In [ ]:
# ── Malta — EDA (Frede) ───────────────────────────────────────────────────
# Malta has 4 cities but ONE NUTS2 region — aggregate before plotting.
MT_GT_COLS = ['gt_airbnb','gt_hotel','gt_flights','gt_attraction_1','gt_attraction_2','gt_attraction_3']
df_malta_region = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in MT_GT_COLS},
          'nights_spent_nuts2': 'first',
          'arrivals_country':   'first',
          'month': 'first'})
)
# TODO (Frede): time-series plot of GT signals vs. nights_spent_nuts2 for Malta,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


In [ ]:
# ── Greece — EDA (Alex / Beatriz) ────────────────────────────────────────
# TODO: time-series plot of GT signals vs. nights_spent_nuts2 for GR cities,
# shock detection markers (>2σ above 3yr rolling mean), correlation heatmap.
pass


---
## 4. Prediction

### 4.1 Why Random Forest?

> **TODO (all):** Justify the model choice here — why Random Forest over linear regression, ARIMA, or gradient boosting for this setting? Consider: non-linearity, mixed feature types (GT signals + seasonal + policy controls), interpretability via SHAP, robustness to missing values.

### 4.2 Feature Engineering


In [ ]:
GT_COLS      = ['gt_airbnb','gt_hotel','gt_flights',
                'gt_attraction_1','gt_attraction_2','gt_attraction_3']
COVID_COLS   = ['dest_stringency_index','origin_top3_stringency_avg']
SEASON_COLS  = ['month','arrivals_last_year','arrivals_yoy_pct_change']
FEATURE_COLS = GT_COLS + COVID_COLS + SEASON_COLS
TARGET       = 'nights_spent_nuts2'

# Malta: aggregate 4 cities → 1 region before modelling
malta_agg = (
    df[df['country']=='Malta']
    .groupby('date', as_index=False)
    .agg({**{c: 'mean' for c in GT_COLS + COVID_COLS},
          **{c: 'first' for c in SEASON_COLS + [TARGET,'country','region']}})
)
malta_agg['city'] = 'malta'

df_model = pd.concat([
    df[df['country'] != 'Malta'],
    malta_agg
], ignore_index=True)

# One-hot encode country to let the model distinguish destinations
df_model = pd.get_dummies(df_model, columns=['country'], drop_first=False)
country_dummies = [c for c in df_model.columns if c.startswith('country_')]
FEATURE_COLS_FULL = FEATURE_COLS + country_dummies

df_model = df_model.dropna(subset=FEATURE_COLS_FULL + [TARGET])
print(f'Model dataset: {df_model.shape[0]} rows, {len(FEATURE_COLS_FULL)} features')
print('Features:', FEATURE_COLS_FULL)


### 4.3 Train / Test Split


In [ ]:
# Time-based split: last 12 months = test (2025-04 → 2026-03)
cutoff = df_model['date'].max() - pd.DateOffset(months=12)
train  = df_model[df_model['date'] <= cutoff]
test   = df_model[df_model['date'] >  cutoff]

X_train = train[FEATURE_COLS_FULL]
y_train = train[TARGET]
X_test  = test[FEATURE_COLS_FULL]
y_test  = test[TARGET]

print(f'Train: {len(train)} rows ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test:  {len(test)} rows ({test["date"].min().date()} → {test["date"].max().date()})')


### 4.4 Random Forest Model


In [ ]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)
print(f'MAE: {mae:,.0f} bed nights')
print(f'R²:  {r2:.3f}')

# Actual vs. predicted
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(y_test, y_pred, alpha=0.4, s=20)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', lw=1)
ax.set_xlabel('Actual nights_spent_nuts2')
ax.set_ylabel('Predicted')
ax.set_title(f'Random Forest — all countries  |  MAE={mae:,.0f}  R²={r2:.3f}')
plt.tight_layout()
plt.show()


### 4.5 SHAP Feature Importance


In [ ]:
# pip install shap  ←  run once if not installed
# !pip install shap -q
import shap

explainer  = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# Global feature importance
shap.summary_plot(shap_values, X_test, plot_type='bar', show=True)

# Beeswarm — direction of effect
shap.summary_plot(shap_values, X_test, show=True)


### 4.6 Per-Country Performance


In [ ]:
# Break down MAE / R² per country on the test set
test_results = test[['date','country','city']].copy()
# Re-apply country dummies for test set lookup
test_results['y_true'] = y_test.values
test_results['y_pred'] = y_pred

# Malta was aggregated — map back
perf = []
for country in sorted(df['country'].unique()):
    mask = test_results['country'] == country
    if mask.sum() == 0:
        continue
    _mae = mean_absolute_error(test_results.loc[mask,'y_true'], test_results.loc[mask,'y_pred'])
    _r2  = r2_score(test_results.loc[mask,'y_true'], test_results.loc[mask,'y_pred'])
    perf.append({'Country': country, 'MAE': round(_mae), 'R²': round(_r2, 3), 'n_test': mask.sum()})

perf_df = pd.DataFrame(perf)
print(perf_df.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
perf_df.plot.bar(x='Country', y='MAE', ax=axes[0], legend=False, color='steelblue')
axes[0].set_title('MAE per country (test set)')
axes[0].set_ylabel('MAE (bed nights)')
perf_df.plot.bar(x='Country', y='R²', ax=axes[1], legend=False, color='seagreen')
axes[1].set_title('R² per country (test set)')
axes[1].axhline(0, color='red', lw=0.8, ls='--')
plt.tight_layout()
plt.show()


---
## 5. Graph Neural Network (GNN)

### 5.1 Why a GNN?

> **TODO:** Justify the GNN — viral tourism shocks don't respect national borders. A destination that goes viral (e.g. Albania via TikTok) can trigger spillover demand to neighbouring regions. A GNN allows us to model these spatial dependencies explicitly by representing destinations as nodes and shared tourist-flow corridors as edges, enabling risk propagation and failure prediction across the network.
>
> Describe graph construction: nodes = destinations, edges = shared origin markets / geographic proximity, edge weights = shared visitor nationality share.

In [ ]:
# ── GNN implementation ────────────────────────────────────────────────────────
# TODO: Implement graph construction + GNN training here.
# Suggested libraries: torch_geometric or spektral (Keras).
pass

---
## 6. Recommender System

### 6.1 Concept & Justification

> **TODO:** Explain the recommender system — what does it recommend, to whom, and why is it the right tool here? Possible framing: given a destination's current digital-signal profile, recommend which *resilience interventions* (capacity limits, diversification campaigns, off-season promotion) are most likely to dampen an incoming demand shock, based on similarity to historical shock-recovery patterns from other destinations.

In [ ]:
# ── Recommender System implementation ─────────────────────────────────────────
# TODO: Implement recommender logic here.
pass

---
## 7. Final Solution System Architecture

> **TODO (all):** Insert the end-to-end system architecture diagram showing how all components connect:
> Digital signals (GT / Wiki) → Feature engineering (seasonality + COVID controls) → Random Forest per region → GNN spillover layer → Recommender System → Resilience dashboard output.

<!-- ![Final architecture](images/final_architecture.png) -->

---
## 8. Conclusion

> **TODO (all):** Summarise main findings:
> - Do GT signals predict arrivals? With what lead time?
> - Which features matter most (SHAP)?
> - What do the GNN spillover results reveal?
> - How useful is the recommender for resilience planning?
>
> Then list limitations and future work.